# JP-1 本番（G1）：日本語・A（Sarashina2.2 1B・float32）

凍結した計画 `notes/jp1_sarashina_gate.md` と変更1（`notes/jp1_amendment1.md`）のとおり、**日本語のA（40問のうち行 0〜19 の20問）** の土台の道と引き直しを作ります。G0 の2回目（`60e611b`）で合格した 1B・float32・先頭1回の引き直し・作業2つのままです。

- 4冊（日本語A・日本語B・英語A・英語B）は **1冊ずつ** 回します。このノートブックが終わってから、次のノートブックをもらってね。
- セル2で、版・精度の設定・G0 の2回目の合格・**作業プロセスのコードが G0 の2回目と同じこと**・問題の番号を確かめます。違えば推論の前に止まります。
- セル2の最後で **Dropbox にサインイン** します（シークレット `DROPBOX_APP_KEY`、あれば `DROPBOX_REFRESH_TOKEN`。なければリンクを開いて「許可」を押し、コードを貼る手順が最初の数分だけ出ます）。
- セル4が本番（作業2つが、終わっていない問題を取りに行きます）、セル5が事後再生監査（この半分の 3問（行 2・10・11））、セル6で結果を Dropbox（`/stash/jp1/ja_A/`）に預けて、終わったらランタイムを切ります。
- **途中で切れたとき**：セル1 → 2 → 3 → 4 → 5 → 6 をもう一度実行（できあがった分は Drive に残っていて、残りだけ計算します）。
- 問題の中身・モデルの答え・トークンは、画面にもファイルにも出しません（数・ID・ハッシュだけ）。

**使い方**：ランタイムは **GPU（L4）**。シークレットは `GITHUB_TOKEN` と `DROPBOX_APP_KEY`（と、あれば `DROPBOX_REFRESH_TOKEN`）。「すべてのセルを実行」で **本番 1〜2 時間くらい＋監査 1 時間くらい**（見込みは G0 の2回目から）。結果は Drive の `DECISION_POINTS/jp1/MAIN/JP1_ja_A/`。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, GPU, exact head, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'JP-1 本番 日本語A（G1・GPU・Sarashina 1B・float32・JA-A）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - JP-1 main JA-A', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'jp1_main_ja_a'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU のランタイム（L4）にしてね"
GPU_NAME = gpu.stdout.strip().split(",")[0]
EXPECTED_HEAD = "27ce60546987b660b0e4942659fccf6ba19b82d4"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
LANG, HALF = "ja", "A"
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")
BASE = THEME / "jp1" / "MAIN" / f"JP1_{LANG}_{HALF}"          # this notebook's run folder
D = BASE
UNITS = BASE / "units"
G0R2 = THEME / "jp1" / "G0_RUN2"                               # G0 run 2 (read only)
G0_HEAD = "60e611bb585011a35138d07e1b10219199039c7f"
LOCKDIR = Path("/content/jp1_locks")
MODEL_DIR = Path("/content/sarashina_1b")
UNITS.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))
STARTED = time.time()
print("GPU:", GPU_NAME, "| head:", head)

In [ ]:
# Cell 2 — pinned packages and checks before anything else: versions, precision (D1), G0 run 2, code identity, items.
from pork import jp1 as P
PINS = [f"{k}=={v}" for k, v in P.PACKAGES.items()] + ["protobuf"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import importlib.metadata as md
CHECKS = {"packages": {k: md.version(k) for k in P.PACKAGES}, "torch": md.version("torch"),
          "gate_doc_sha256_ok": P.gate_doc_sha256(REPO) == P.GATE_DOC_SHA256,
          "lock_file_present": (REPO / "notes/jp1_lock_pre_g0.md").exists(),
          "amendment1_present": (REPO / P.AMENDMENT1_RELPATH).exists()}
import torch as torch_mod
CHECKS["precision_flags"] = P.precision_flags(torch_mod)
CHECKS["torch_cuda"] = str(torch_mod.version.cuda)
CHECKS["gpu"] = GPU_NAME

# G0 run 2 passed (1B, all prechecks, M1, time and CU lines) on this GPU type, at G0_HEAD
G0S = json.loads((G0R2 / "G0_SUMMARY.json").read_text(encoding="utf-8"))
G0C = json.loads((G0R2 / "G0_CHECKS.json").read_text(encoding="utf-8"))["checks"]
CHECKS["g0_run2"] = {"head_ok": G0S["head"] == G0_HEAD, "chosen": G0S["chosen_model"], "prechecks_pass": G0S["prechecks_pass"],
                     "parity_pass": G0S["parity_pass"], "m1_pass": G0S["amendment1"]["M1"]["pass"],
                     "per_notebook_ok": G0S["timing"]["per_notebook_ok"], "total_cu_ok": G0S["timing"]["total_cu_ok"],
                     "same_gpu": G0S["gpu"] == GPU_NAME}
MODEL_KEY = G0S["chosen_model"]
G0_OK = (CHECKS["g0_run2"]["head_ok"] and MODEL_KEY in P.MODELS and all(v is True for k, v in CHECKS["g0_run2"].items()
                                                                       if k not in ("head_ok", "chosen")))

# the JP-1 code the workers import is byte-identical to G0 run 2's
_imp = ("import sys, pork.jp1, pork.jp1_text, pork.jp1_runner, pork.jp1_shared_prefix, pork.jp1_reference, "
        "pork.kv_cached_backend, pork.pork_t_v1, pork.ct1_runner, pork.quality_pork_q3a_hightemp, pork.q4b_samplers, "
        "pork.dp1_forks_audit, pork.ct1_forks_replacement, pork.dp1_forks_runner; "
        "print(chr(10).join(sorted(m.__file__ for n, m in list(sys.modules.items()) "
        "if n.split('.')[0] == 'pork' and getattr(m, '__file__', None))))")
_files = subprocess.check_output([sys.executable, "-c", _imp], cwd=REPO, text=True).split()
CODE_FILES = sorted(str(Path(f).resolve().relative_to(REPO)) for f in _files)
CHECKS["code_files"] = len(CODE_FILES)
CHECKS["code_same_as_g0_run2"] = subprocess.run(["git", "-C", str(REPO), "diff", "--quiet", G0_HEAD, EXPECTED_HEAD, "--",
                                                 *CODE_FILES]).returncode == 0

# MGSM items as in G0 run 2 (hashes only)
ROWS = {L: P.load_rows(L) for L in ("ja", "en")}
CHECKS["pairing"] = P.check_pairing(ROWS["ja"], ROWS["en"])
ITEMS = P.items(ROWS[LANG], "main")
del ROWS
ITEMS_SHA = P.items_sha256(ITEMS)
CHECKS["items_sha256"] = ITEMS_SHA
CHECKS["items_same_as_g0_run2"] = ITEMS_SHA == G0C["items_sha256"][LANG]
VERSION_OK = (CHECKS["gate_doc_sha256_ok"] and CHECKS["lock_file_present"] and CHECKS["amendment1_present"]
              and CHECKS["packages"] == P.PACKAGES and CHECKS["torch"].split("+")[0] == P.TORCH_BASE_VERSION
              and P.precision_ok(CHECKS["precision_flags"]) and CHECKS["pairing"]["pass"]
              and CHECKS["items_same_as_g0_run2"])
from pork.quality_pork_q3a_runner import atomic_json
atomic_json(BASE / "MAIN_CHECKS.json", {"head": EXPECTED_HEAD, "g0_head": G0_HEAD, "checks": CHECKS, "code_files": CODE_FILES,
                                        "version_ok": VERSION_OK, "g0_ok": G0_OK})
print(json.dumps({k: v for k, v in CHECKS.items() if k != "items_sha256"}, indent=1, ensure_ascii=False))
assert VERSION_OK, P.STOP_VERSION + " — 推論の前に止めました。Claude に見せてね。"
assert G0_OK, "G0 の2回目の合格の記録と合いません — Claude に見せてね"
assert CHECKS["code_same_as_g0_run2"], "作業プロセスのコードが G0 の2回目と違います — Claude に見せてね"
print("CHECKS OK — model", MODEL_KEY, "| code files", len(CODE_FILES))
# Dropbox (the Kaggle CPU analysis reads the results from here).  Signing in now, so nobody has to be there at the end.
from notebook_tools import dbx
def _secret(n):
    try:
        return userdata.get(n) or None
    except Exception:
        return None
DBX_KEY, DBX_RT = _secret("DROPBOX_APP_KEY"), _secret("DROPBOX_REFRESH_TOKEN")
print("DROPBOX_APP_KEY", "あり" if DBX_KEY else "★なし★", "/ DROPBOX_REFRESH_TOKEN", "あり" if DBX_RT else "なし（ここで作ります）")
if not DBX_KEY:
    raise RuntimeError("Colab のシークレットに DROPBOX_APP_KEY（Kaggle に入れたのと同じ値）を追加して、このノートブックで有効にしてください")
if DBX_RT:
    try:
        dbx.Dropbox(DBX_KEY, DBX_RT).token()
    except dbx.DropboxError as e:
        print("★ DROPBOX_REFRESH_TOKEN が使えませんでした（", type(e).__name__, "）。ここで作り直します ★"); DBX_RT = None
if not DBX_RT:
    _verifier, _url = dbx.pkce_start(DBX_KEY)
    print("① このリンクを開いて、Dropbox で「許可」を押してください："); print(_url)
    DBX_RT = dbx.pkce_finish(DBX_KEY, _verifier, input("② 表示されたコードをここに貼って Enter：").strip())
    print("③ 次からこの手順を飛ばすには、Colab のシークレット DROPBOX_REFRESH_TOKEN に今の値を入れてください（人に見せないでね）")
DBX = dbx.Dropbox(DBX_KEY, DBX_RT)
STASH = f"/stash/jp1/{LANG}_{HALF}"
DBX.upload_bytes((json.dumps({"code_commit": EXPECTED_HEAD, "lang": LANG, "half": HALF, "started_jst":
                              dt.datetime.now(JST).isoformat(timespec="seconds")}) + "\n").encode(), STASH + "/_start.json")
print("Dropbox に書けました（アプリ用フォルダの", STASH, "）")

In [ ]:
# Cell 3 — the model at the pinned revision on local disk; RUN_IDENTITY (new, or the same when resuming); unit files on
# Drive validated before any worker starts (an unreadable one is moved aside and recomputed).
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoConfig, GenerationConfig
from types import SimpleNamespace
from pork import kv_cached_backend as kv
from pork import jp1_runner as rr
SPEC = P.MODELS[MODEL_KEY]
snapshot_download(SPEC["id"], revision=SPEC["revision"], local_dir=str(MODEL_DIR), token=False,
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.model"])
_tk = AutoTokenizer.from_pretrained(str(MODEL_DIR))
_vocab = sorted(_tk.get_vocab().items())
assert hashlib.sha256(json.dumps(_vocab, ensure_ascii=False).encode("utf-8")).hexdigest() == P.TOKENIZER_VOCAB_SHA256
del _vocab
try:
    _gc = GenerationConfig.from_pretrained(str(MODEL_DIR))
except Exception:
    _gc = GenerationConfig.from_model_config(AutoConfig.from_pretrained(str(MODEL_DIR)))
EOS = kv.eos_ids(_tk, SimpleNamespace(generation_config=_gc))
IDENTITY = {"code_commit": EXPECTED_HEAD, "lang": LANG, "half": HALF, "model": MODEL_KEY, "model_revision": SPEC["revision"],
            "model_local_path": str(MODEL_DIR), "dtype": SPEC["dtype"], "items_sha256": ITEMS_SHA, "eos_ids": list(EOS),
            "gpu_name": GPU_NAME, "g0_head": G0_HEAD, "amendment": P.AMENDMENT1_RELPATH,
            "rows": P.HALVES[HALF], "max_gpu_workers": P.MAX_GPU_WORKERS}
id_path = BASE / "RUN_IDENTITY.json"
if id_path.exists():
    _old = json.loads(id_path.read_text(encoding="utf-8"))
    _diff = sorted(k for k in IDENTITY if _old.get(k) != IDENTITY[k])
    assert not _diff, ("前回の RUN_IDENTITY と違います（続きから再開できません）— Claude に見せてね", _diff)
    print("RESUME: the same run identity")
else:
    atomic_json(id_path, IDENTITY)
    print("new run identity written")

# validate unit files already on Drive (after a disconnect a file could be cut off)
QUAR = BASE / "quarantine"
moved = []
for _p in sorted(UNITS.glob("*.json")):
    _parts = _p.stem.split("_")
    _kind, _row = _parts[0], int(_parts[1])
    _i = int(_parts[2]) if _kind == "point" else None
    try:
        rr.read_envelope(_p, LANG, MODEL_KEY, _kind, _row, _i, EXPECTED_HEAD, ITEMS_SHA, tries=1)
    except Exception as _e:
        QUAR.mkdir(exist_ok=True)
        _dst = QUAR / f"{_p.stem}_{int(time.time())}.json"
        _p.rename(_dst)
        moved.append({"file": _p.name, "moved_to": _dst.name, "error": type(_e).__name__})
if moved:
    _ql = BASE / "QUARANTINE.json"
    atomic_json(_ql, (json.loads(_ql.read_text()) if _ql.exists() else []) + [{"unix": time.time(), "moved": moved}])
def n_units():
    return len(list(UNITS.glob("base_*.json"))), len(list(UNITS.glob("point_*.json")))
print(json.dumps({"model": MODEL_KEY, "eos_ids": list(EOS), "units_on_drive": n_units(), "moved_aside": len(moved)}))

In [ ]:
# Cell 4 — BENCHMARK: 2 worker processes of pork.jp1_runner pull the unfinished units of this half (local lock files; a
# unit's file depends only on its row and point, so the order cannot change any file).  The notebook holds no GPU memory.
# Resumable: after a disconnect, rerun cells 1-4 — finished units stay on Drive and are not recomputed.
import signal
for _m in BASE.glob("RUN_FAILED*.txt"):
    raise SystemExit(f"{_m.name} があります（PROTOCOL_FAIL）。続けずに Claude に見せてね")
_smi = subprocess.run(["nvidia-smi", "--query-gpu=memory.free,memory.total", "--format=csv,noheader,nounits"],
                      capture_output=True, text=True, check=True).stdout.strip().splitlines()[0]
_free, _total = (int(x) * 1.048576e6 / 1e9 for x in _smi.split(","))   # MiB -> GB (the notebook never touches CUDA)
CHECKS_GPU = {"free_gb_before": _free, "total_gb": _total}
assert _free > _total - 1.0, f"このノートブックが GPU を使っています（{_total - _free:.1f} GB）— Claude に見せてね"
WORKERS = max(1, min(rr.MAX_WORKERS, P.MAX_GPU_WORKERS, kv.physical_cores(), int(_total // 10)))
assert WORKERS == P.MAX_GPU_WORKERS, f"作業プロセスが {WORKERS} つしか立てられません（M1 は2つで確かめた）— Claude に見せてね"
HALF_ROWS = len(P.HALVES[HALF])

def live_worker_pids():
    r = subprocess.run(["pgrep", "-f", f"pork.jp1_runner --lang {LANG} --half {HALF}"], capture_output=True, text=True)
    return [int(x) for x in r.stdout.split()]

def pid_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

old = live_worker_pids()
procs = {}
if old:
    print("ℹ️ 前に起動した作業プロセスがまだ動いています（pid " + str(old) + "）。二重起動せず、終わるまで見守ります。")
else:
    shutil.rmtree(LOCKDIR, ignore_errors=True)   # no live worker: every lock is stale
    LOCKDIR.mkdir(parents=True, exist_ok=True)
    for w in range(WORKERS):
        logf = BASE / f"worker_{w}.log"
        procs[w] = subprocess.Popen(
            [sys.executable, "-m", "pork.jp1_runner", "--lang", LANG, "--half", HALF, "--worker", str(w), "--base", str(BASE),
             "--lockdir", str(LOCKDIR), "--head", EXPECTED_HEAD],
            cwd=REPO, stdout=open(logf, "a"), stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
        print(f"worker {w}: pid {procs[w].pid}", flush=True)
        time.sleep(20)

def stop_all(why):
    print("🛑", why, "— 作業プロセスを止めます。Claude に見せてね", flush=True)
    for p in procs.values():
        if p.poll() is None:
            p.send_signal(signal.SIGTERM)
    for pid in old:
        if pid_alive(pid):
            os.kill(pid, signal.SIGTERM)

t0 = time.time()
failed = None
STALL_MIN = 60
last_change, last_counts = time.time(), n_units()
while True:
    running = [w for w, p in procs.items() if p.poll() is None] + [pid for pid in old if pid_alive(pid)]
    bad = [w for w, p in procs.items() if p.poll() not in (None, 0)]
    if any(BASE.glob("RUN_FAILED*.txt")) or bad:
        failed = "worker failure (" + str(bad or sorted(p.name for p in BASE.glob("RUN_FAILED*.txt"))) + ")"
        stop_all(failed); break
    if not running:
        break
    time.sleep(60)
    nb, npnt = n_units()
    if (nb, npnt) != last_counts:
        last_change, last_counts = time.time(), (nb, npnt)
    elif time.time() - last_change > STALL_MIN * 60:
        failed = f"no unit finished for {STALL_MIN} min (stalled)"
        stop_all(failed); break
    print(f"{(time.time() - t0) / 60:6.1f} min  base {nb}/{HALF_ROWS}  points {npnt}  running {len(running)}", flush=True)
for w, p in procs.items():
    p.wait()
    print(f"worker {w}: exit code {p.returncode}")
    print((BASE / f"worker_{w}.log").read_text(errors="replace")[-600:])
assert failed is None, failed
CASES = rr.load_cases(LANG, MODEL_KEY, P.HALVES[HALF], ITEMS, units_dir=UNITS, code_commit=EXPECTED_HEAD, items_hash=ITEMS_SHA)
RUN_HOURS = (time.time() - t0) / 3600
nb, npnt = n_units()
assert nb == HALF_ROWS and npnt == sum(len(c["points"]) for c in CASES), (nb, npnt)
print("complete: base", nb, "/", HALF_ROWS, "| points", npnt, f"| {RUN_HOURS:.2f} h  👉 セル5（監査）へ")

In [ ]:
# Cell 5 — post-hoc replay audit (plan G2, as DP5): this half's pre-declared rows, the base path and 3 points recomputed
# with samples j = 0..3 drawn by the plain-Python reference; same GPU type and the same shared-prefix step (amendment 1).
# The pass rule (per language, at most 1 diverged row over both halves) is applied in the analysis (Kaggle).
assert not any(BASE.glob("RUN_FAILED*.txt")), "failure markers exist — show Claude"
AUDIT_ROWS = rr.audit_rows_in_half(HALF)
audit_path = BASE / f"AUDIT_{LANG}_{HALF}.json"
t1 = time.time()
if not audit_path.exists():
    logf = BASE / "audit.log"
    p = subprocess.run([sys.executable, "-m", "pork.jp1_runner", "--lang", LANG, "--half", HALF, "--worker", "99",
                        "--base", str(BASE), "--lockdir", str(LOCKDIR), "--head", EXPECTED_HEAD, "--audit"],
                       cwd=REPO, stdout=open(logf, "a"), stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    print(logf.read_text(errors="replace")[-600:])
    assert p.returncode == 0, "監査の実行に失敗しました — Claude に見せてね"
AUDIT_HOURS = (time.time() - t1) / 3600
AUD = json.loads(audit_path.read_text(encoding="utf-8"))
assert AUD["rows"] == AUDIT_ROWS, (AUD["rows"], AUDIT_ROWS)
print(json.dumps({"rows": AUD["rows"], "diverged_rows": AUD["diverged_rows"], "hours": round(AUDIT_HOURS, 2)}))

In [ ]:
# Cell 6 — hand-off to Dropbox for the Kaggle CPU analysis (units + identity + audit as one tar.gz, content-hash checked,
# then _done.json), the summary (numbers only), notify and disconnect.
import tarfile
TAR = Path(f"/content/jp1_{LANG}_{HALF}.tar.gz")
_members = ["RUN_IDENTITY.json", "MAIN_CHECKS.json", f"AUDIT_{LANG}_{HALF}.json", "units"]
with tarfile.open(TAR, "w:gz") as tf:
    for _m in _members:
        tf.add(str(BASE / _m), arcname=_m)
TAR_SHA = hashlib.sha256(TAR.read_bytes()).hexdigest()
DBX.upload(str(TAR), STASH + "/" + TAR.name)                      # raises if Dropbox's content hash differs
nb, npnt = n_units()
DONE = {"code_commit": EXPECTED_HEAD, "lang": LANG, "half": HALF, "model": MODEL_KEY, "file": TAR.name,
        "sha256": TAR_SHA, "bytes": TAR.stat().st_size, "units": {"base": nb, "point": npnt},
        "audit_rows": AUD["rows"], "audit_diverged_rows": AUD["diverged_rows"],
        "finished_jst": dt.datetime.now(JST).isoformat(timespec="seconds")}
DBX.upload_bytes((json.dumps(DONE) + "\n").encode(), STASH + "/_done.json")
SUMMARY = {"schema": "jp1_main_v1", **DONE, "gpu": GPU_NAME, "workers": WORKERS, "run_hours": RUN_HOURS,
           "audit_hours": AUDIT_HOURS, "elapsed_hours": (time.time() - STARTED) / 3600, "checks_gpu": CHECKS_GPU,
           "dropbox": STASH, "g0_head": G0_HEAD}
atomic_json(BASE / "MAIN_SUMMARY.json", SUMMARY)
print(json.dumps({k: SUMMARY[k] for k in ("lang", "half", "units", "audit_diverged_rows", "run_hours", "audit_hours", "sha256")}, indent=1))
notify("本番 日本語A 終わりました（Dropbox に預けました）", tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()